# LayoutLMv3 Fine-Tuning Pipeline (Google Colab & Local)

This notebook handles GPU training setup, dynamic class-weighted cross-entropy loss computation, and fine-tuning LayoutLMv3 for 50 epochs over `Training_layoutLMV3.json` to produce the trained `model.bin` checkpoint.


## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Required Libraries

In [ ]:
!pip install transformers seqeval accelerate pillow

## Step 3: Verify GPU Availability

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
!nvidia-smi

## Step 4: Set Working Directory

In [ ]:
import os
PROJECT_DIR = '/content/drive/MyDrive/LayoutLMv3-FT'
os.chdir(PROJECT_DIR)
print("Current working directory:", os.getcwd())
!ls -la

## Step 5: Interactive Training Pipeline with Dynamic Class-Weighted Loss

This cell counts the frequencies of all class labels in `Training_layoutLMV3.json` dynamically and computes inverse frequency weights. This penalizes the model heavier for missing rare fields (like `amount_due` and `vendor_phone`), raising recall without any hardcoding.

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as nnf
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from tqdm.notebook import tqdm
from transformers import LayoutLMv3ImageProcessor, LayoutLMv3TokenizerFast, LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from seqeval.metrics import precision_score, recall_score, f1_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1. LOAD CONFIGS AND MAPPINGS
with open("label_config.json", "r") as f:
    config = json.load(f)
    label2id = config["label2id"]
    id2label = {int(v): k for k, v in config["label2id"].items()}

# 2. DEFINE PYTORCH DATASET
class LayoutLMv3Dataset(Dataset):
    def __init__(self, json_path, processor):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.processor = processor
        self.features = []
        
        print("Pre-tokenizing and chunking dataset with sliding window...")
        for item in self.data:
            img_name = os.path.basename(item["file_name"])
            img_path = os.path.join("images", img_name)
            
            if not os.path.exists(img_path):
                # Fallback for folder paths inside colab if running from subdirectories
                img_path = os.path.join("Finetuning", "inputs", "images", img_name)
            if not os.path.exists(img_path):
                raise FileNotFoundError(f"Could not find image at {img_name}")

            image = Image.open(img_path).convert("RGB")
            
            tokens = []
            boxes = []
            labels = []
            
            for ann in item["annotations"]:
                tokens.append(ann["text"])
                boxes.append(ann["box"])
                labels.append(label2id.get(ann["label"], 0))

            encoding = self.processor(
                image,
                tokens,
                boxes=boxes,
                word_labels=labels,
                max_length=512,
                stride=128,
                padding="max_length",
                truncation=True,
                return_overflowing_tokens=True,
                return_tensors="pt"
            )
            
            for i in range(len(encoding.input_ids)):
                self.features.append({
                    "input_ids": encoding.input_ids[i],
                    "attention_mask": encoding.attention_mask[i],
                    "bbox": encoding.bbox[i],
                    "pixel_values": encoding.pixel_values[i],
                    "labels": encoding.labels[i]
                })
        print(f"Successfully generated {len(self.features)} chunks from {len(self.data)} documents.")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]

class ModelModule(nn.Module):
    def __init__(self, num_classes, class_weights=None):
        super().__init__()
        self.model = LayoutLMv3ForTokenClassification.from_pretrained(
            "microsoft/layoutlmv3-base", 
            num_labels=num_classes, 
            ignore_mismatched_sizes=True
        )
        self.num_labels = num_classes
        if class_weights is not None:
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        else:
            self.register_buffer("class_weights", torch.tensor([1.0] * num_classes, dtype=torch.float32))

    def forward(self, input_ids, attention_mask, bbox, pixel_values, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values
        )
        logits = outputs.logits
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights, ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return logits, loss

# 4. INITIALIZE PROCESSOR, DATASET AND DATALOADER
feature_extractor = LayoutLMv3ImageProcessor(apply_ocr=False)
tokenizer = LayoutLMv3TokenizerFast.from_pretrained("microsoft/layoutlmv3-base")
processor = LayoutLMv3Processor(tokenizer=tokenizer, image_processor=feature_extractor)

train_dataset = LayoutLMv3Dataset("Training_layoutLMV3.json", processor)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# 4B. DYNAMICALLY COMPUTE CLASS WEIGHTS (INVERSE FREQUENCY WITH CLIPPING)
from collections import Counter
label_counts = Counter()
for item in train_dataset.data:
    for ann in item["annotations"]:
        label_name = ann["label"]
        label_id = label2id.get(label_name, 0)
        label_counts[label_id] += 1

total_instances = sum(label_counts.values())
num_classes = len(label2id)
computed_weights = [1.0] * num_classes

# Calculate dynamic weights inversely proportional to target class counts
for label_id in range(num_classes):
    if label_id == 0:  # Keep baseline weight for "O" class (outside) at 1.0 to maintain stability
        continue
    count = label_counts.get(label_id, 0)
    if count > 0:
        # Inverse frequency weight relative to class representation
        weight = total_instances / (num_classes * count)
        # Keep weights numerically stable by capping between 1.0 and 10.0
        computed_weights[label_id] = min(max(weight, 1.0), 10.0)

print("Dynamically Computed Class Weights:")
for label_id, w in enumerate(computed_weights):
    if w > 1.0:
        print(f"  Class {label_id:<2} ({id2label[label_id]:<30}): Weight = {w:.2f}")

model = ModelModule(len(label2id), class_weights=computed_weights)
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 50
best_loss = np.inf

# 5. TRAINING LOOP
print("--- STARTING TRAINING LOOP ---")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        bbox = batch["bbox"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        logits, loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values,
            labels=labels
        )
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Complete. Average Loss: {avg_loss:.4f}")
    
    # Save best model weight check
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "model.bin")
        print("  Saved new best checkpoint to 'model.bin'")